In [ ]:
from pyspark.sql.types import *
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [ ]:
spark = (SparkSession
       .builder
       .appName("DF SF Fire Example")
       .getOrCreate())

# Reading structured data using a schema 

Spark can infer the schema but that can be costly. Spark can also infer schema from a sample at a lesser cost. Here we will define the schema explicitly. Note that we changed column names to not have space in them as that can be problematic when saving data.

In [ ]:
fire_schema =  StructType([StructField('CallNumber', IntegerType(), True),
                           StructField('UnitID', StringType(), True),
                           StructField('IncidentNumber', IntegerType(), True),
                           StructField('CallType', StringType(), True),
                           StructField('CallDate', StringType(), True),
                           StructField('WatchDate', StringType(), True),
                           StructField('CallFinalDisposition', StringType(), True),
                           StructField('AvailableDtTm', StringType(), True),
                           StructField('Address', StringType(), True),
                           StructField('City', StringType(), True),
                           StructField('Zipcode', IntegerType(), True),
                           StructField('Battalion', StringType(), True),
                           StructField('StationArea', StringType(), True),
                           StructField('Box', StringType(), True),
                           StructField('OriginalPriority', StringType(), True),
                           StructField('Priority', StringType(), True),
                           StructField('FinalPriority', IntegerType(), True),
                           StructField('ALSUnit', BooleanType(), True),
                           StructField('CallTypeGroup', StringType(), True),
                           StructField('NumAlarms', IntegerType(), True),
                           StructField('UnitType', StringType(), True),
                           StructField('UnitSequenceInCallDispatch', IntegerType(), True),
                           StructField('FirePreventionDistrict', StringType(), True),
                           StructField('SupervisorDistrict', StringType(), True),
                           StructField('Neighborhood', StringType(), True),
                           StructField('Location', StringType(), True),
                           StructField('RowID', StringType(), True),
                           StructField('Delay', FloatType(), True)])

In [ ]:
sf_fire_file = "sf-fire/sf-fire-calls.csv"
fire_df = spark.read.csv(sf_fire_file, header=True, schema=fire_schema)

# Projections and Filters

A *projection* in relation parlance is a way to return only the rows matching a certain realtional condition by using filters. In Spark, projections are done with the `select()` method, while filters can be expressed using the `filter()` or `where()` methods.

In [ ]:
few_fire_df = (fire_df.select("IncidentNumber", "AvailableDtTm", "CallType")
                .where(col("CallType") != "Medical Incident"))

In [ ]:
few_fire_df.show(5,False)

What if we want to know how many distinct `CallTypes` were recorded as the causes of fire calls?

In [ ]:
(fire_df
 .select("CallType")
 .where(col("CallType").isNotNull())
 .agg(count_distinct("CallType").alias("DistinctCallTypes"))
 .show(truncate=False))
      

We can list the distinct call types in the daya set using the following query.

In [ ]:
(fire_df
 .select("CallType")
 .where(col("CallType").isNotNull())
 .distinct()
 .show(truncate=False))

# Renaming, Adding, and Dropping Columns

We can rename columns for reasons of convetion, or sometimes for readability or brevity. Let's change the name of our `Delayed` column to `ResponseDelayedinMins` and take a look at response times that were longer than five minutes.

In [ ]:
fire_df.cache()

In [ ]:
new_fire_df = fire_df.withColumnRenamed("Delay", "ResponseDelayedinMins")

In [ ]:
(new_fire_df
 .select("ResponseDelayedinMins")
 .where(col("ResponseDelayedinMins") > 5)
 .show(5, False))

In [ ]:
fire_ts_df = (new_fire_df
              .withColumn("IncidentDate", to_timestamp(col("CallDate"), "MM/dd/yyyy"))
              .drop("CallDate")
              .withColumn("OnWatchDate", to_timestamp(col("WatchDate"), "MM/dd/yyyy"))
              .drop("WatchDate")
              .withColumn("AvailableDtTS", to_timestamp(col("AvailableDtTm"), "MM/dd/yyyy hh:mm:ss a"))
              .drop("AvailableDtTm"))
              

In [ ]:
(fire_ts_df
 .select("IncidentDate", "OnWatchDate", "AvailableDtTS")
         .show(5, False))

Now we can query using functions from `spark.sql.functions` like `dayofmonth(), dayofyear, dayofweek()` to explore. For example, we could find out how many calls were logged in the last seven days, or we could see how many years' worth of calls are included in the data set.

In [ ]:
(fire_ts_df
  .select(year('IncidentDate'))
 .distinct()
 .orderBy(year('IncidentDate'))
 .show())

# Aggregations

What if we want to know what the most common types of fire calls were, or what zip codes accounted for the most calls?

A handful of transformations and actions on DataFrames such as `groupBy()`, `orderBy()`, and `count()`, offer the ability to aggregate by column names and then aggregate counts across them.

In [ ]:
(fire_ts_df
 .select("callType")
 .where(col("CallType").isNotNull())
 .groupBy("callType")
 .count()
 .orderBy("count", ascending=False)
 .show(n=10, truncate=False))

DataFrames also provide common statistical functions such as `min(), max(), sum()`, and `avg()`. Here is an example. 

In [ ]:
import pyspark.sql.functions as F

In [ ]:
(fire_ts_df
 .select(F.sum("NumAlarms"), F.avg("ResponseDelayedinMins"),
         F.min("ResponseDelayedinMins"), F.max("ResponseDelayedinMins"))
 .show())

Other common statistical methods include `stat()`, `describe()`, `correlation()`, 
`covariance()`, `sampleBy()`, `approxQuantile()`, `frequentItems()`, etc.

# Further Data Explorations

* What were all the different types of fire calls in 2018?
* What months within the year 2018 saw the highest number of fire calls?
* Which neighborhood in San Francisco generated the most fire calls in 2018?
* Which neighborhoods had the worst response times to fire calls in 2018?
* Which week in 2018 had the most fire calls?
* Is there a correlation between neighborhood, zip code, and number of fire calls?

Finally, we show how to write the DataFrame out. The parquet format is a common format that is efficient. It stores the schema as metadata, so we don't need to define it when we read the DataFrame back.

In [ ]:
(fire_df.write.format("parquet")
 .mode("overwrite")
 .option("compression", "snappy")
 .save("./parquet"))

In [ ]:
saved_fire_df = spark.read.format("parquet").load("./parquet")

In [ ]:
saved_fire_df.show(2)

In [ ]:
spark.stop()